In [ ]:
import extractor
import torch
import skimage
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from torchvision.transforms import ToTensor

In [ ]:
PATCH_SIZE = 512
STRIDE = 8
FEATURE_SIZE = 384
TOKENS_PER_PATCH = PATCH_SIZE // STRIDE

DIR = '/home/caicedo/scr/jcaicedo/Micronuclei-data/'

In [ ]:
device = 'cuda:6' if torch.cuda.is_available() else 'cpu'
ex = extractor.ViTExtractor('dino_vits8', STRIDE, device=device)

In [ ]:
def patch_to_rgb(patch):
    px = np.concatenate(
        (patch[:,:,np.newaxis], patch[:,:,np.newaxis], patch[:,:,np.newaxis]), 
        axis=2)
    return ToTensor()(px)

In [ ]:
im = skimage.io.imread(DIR + "20X_c0-DAPI_A1_Tile-110.phenotype.tif")
im = skimage.exposure.rescale_intensity(im, out_range=np.float32)
H,W = im.shape

In [ ]:
w_blocks = int(W/PATCH_SIZE)
h_blocks = int(H/PATCH_SIZE)

In [ ]:
A = []
coords = []
for i in tqdm(range(0,im.shape[0]-PATCH_SIZE,PATCH_SIZE)):
    for j in range(0,im.shape[1]-PATCH_SIZE,PATCH_SIZE):
        rgb = patch_to_rgb(im[i:i+PATCH_SIZE,j:j+PATCH_SIZE])
        ptensor = ex.preprocess_patch(rgb)
        feat = ex.extract_descriptors(ptensor[None,:,:,:].to(device), 11, 'key', False)
        A.append(feat.detach().cpu().numpy())
        coords.append({"x":j,"y":i})

In [ ]:
ptensor.shape

In [ ]:
all_features = np.concatenate(A)
all_features = np.reshape(all_features, (w_blocks, h_blocks, TOKENS_PER_PATCH, TOKENS_PER_PATCH, FEATURE_SIZE))
all_features = np.transpose(all_features, axes=(0,2,1,3,4))
all_features = np.reshape(all_features, (w_blocks*(TOKENS_PER_PATCH), h_blocks*(TOKENS_PER_PATCH), FEATURE_SIZE))
all_features.shape

In [ ]:
plt.imshow(all_features[:,:,0])

In [ ]:
plt.imshow(im)